# Bonus 02 — RAG with LlamaIndex
**Optional | After Lab 6 | Colab CPU | OpenAI API key**

Lab 6 built RAG by hand: chunk, MiniLM, Chroma, grounded prompt. LlamaIndex does the same pipeline in a few lines. The point is not “LlamaIndex is better.” The point is to **feel the abstraction** so you can choose it on purpose.

> **Cost note:** this notebook uses LlamaIndex defaults — **OpenAI embeddings + chat**. Lab 6 used free local MiniLM. You will pay for embedding the sample docs once.

```
Lab 6 (explicit)          This bonus (abstracted)
chunk splitter       →    VectorStoreIndex.from_documents
MiniLM + Chroma      →    default OpenAI embeddings + in-memory store
rag() prompt         →    as_query_engine()
```


In [ ]:
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Python {sys.version.split()[0]}")
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")

if IN_COLAB:
    !pip install -q uv
    !uv pip install -q --system llama-index openai
elif shutil.which("uv"):
    !uv pip install -q llama-index openai
else:
    !pip install -q llama-index openai

print("Install complete")


In [ ]:
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. Colab: key icon → Add secret {name} → enable Notebook access."
        )
    return value

OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Config loaded — {DEFAULT_MODEL}")


## 1. A tiny course corpus (no PDF required)

Colab does not clone `Bonus/sample_docs/` for you. We write four short markdown files if they are missing, then load every file in that folder.


In [ ]:
from pathlib import Path

SAMPLE = Path("sample_docs")
SAMPLE.mkdir(exist_ok=True)
docs_inline = {
    "quantization.md": "Quantization reduces weight precision. NF4 is 4-bit for LLM weights and uses less VRAM than FP16.",
    "rag.md": "RAG retrieves chunks at inference time and asks the model to answer only from that context. Use it for changing knowledge.",
    "lora.md": "LoRA trains small adapter matrices on a frozen base. QLoRA adds a 4-bit base so fine-tuning fits a smaller GPU.",
    "serving.md": "OpenAI-compatible APIs swap backends via base_url. vLLM adds PagedAttention and continuous batching for throughput.",
}
for name, text in docs_inline.items():
    p = SAMPLE / name
    if not p.exists():
        p.write_text(text)
print("files:", sorted(p.name for p in SAMPLE.iterdir()))


## 2. Load → index → query

`SimpleDirectoryReader` picks a parser per file type. `VectorStoreIndex.from_documents` chunks, embeds, and stores. `as_query_engine` is retrieve + generate.


In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(str(SAMPLE)).load_data()
print("Loaded", len(documents), "documents")

index = VectorStoreIndex.from_documents(documents)
print("Index built (OpenAI embeddings — this is the paid step)")

query_engine = index.as_query_engine()
response = query_engine.query("When should I use RAG instead of fine-tuning?")
print(response)


### Inspect the retrieved nodes

Lab 6 printed Chroma distances. LlamaIndex gives `source_nodes` with a score and file name. Always look — that is how you debug a bad answer.


In [ ]:
for node in response.source_nodes:
    print("---")
    print("score:", round(float(node.score or 0), 3))
    print("file :", node.metadata.get("file_name"))
    print(node.text[:220])


## 3. Persist so you do not re-embed

In-memory indexes die with the kernel. Persist once; load later without paying for embeddings again.


In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

index.storage_context.persist(persist_dir="./vector_db_llama")
print("saved ./vector_db_llama")

loaded = load_index_from_storage(StorageContext.from_defaults(persist_dir="./vector_db_llama"))
print(loaded.as_query_engine().query("What is QLoRA?"))


**Checkpoint:** the QLoRA answer should mention adapters / 4-bit. If it talks about byte-pair encoding, you pointed at the wrong folder.

## Student challenge

Upload 1–2 of **your** PDFs into `student_pdfs/` (Colab: folder icon → upload), then rebuild.


In [ ]:
# TODO: load student_pdfs, build an index, ask one question about YOUR docs.
from pathlib import Path
print("student_pdfs exists:", Path("student_pdfs").exists())
# documents = SimpleDirectoryReader("student_pdfs").load_data()


## Bonus 02 complete

LlamaIndex hid chunking and the vector store. Lab 6 made you own them. Use LlamaIndex when the default pipeline is enough; drop to Lab 6 when you need to change chunk size, embeddings, or hybrid search.

Next: [Bonus 05](05_hf_spaces_deployment.md) to persist the Lab 7 Gradio app, or back to [Lab 7](../07_Gradio_RAG_App/README.md).
